# ESZA019 — Visão Computacional
## Laboratório 7 — Introdução às Redes CNN
*Convolutional Neural Networks* para Reconhecimento em Imagens

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** _(preencher — ex.: 24/08/2026)_
**Data de publicação do relatório:** _(preencher)_

---
> **Declaração de uso de IA Generativa (Portaria CNPq nº 2664/2026, item c).**
> Na elaboração deste relatório utilizou-se a ferramenta de IAG **Claude (Anthropic)** para **apoio à
> redação da fundamentação teórica, organização do texto e comentários do código**. Todo o conteúdo foi
> revisado, executado e validado pelos autores, responsáveis integrais pelo material (itens d e f).

> **Como executar.** Este notebook foi projetado para o **Google Colab** com **GPU T4**
> (`Ambiente de execução → Alterar tipo de ambiente de execução → GPU`). As células de código estão
> prontas para rodar de cima para baixo. Onde se lê **_(inserir saída/print/imagem da sua execução)_**,
> cole o resultado obtido na sua máquina/turma (prints, gráficos e fotos da webcam).

## Sumário
1. Introdução
2. Fundamentação Teórica
3. Procedimentos Experimentais (com código)
4. Análise e Discussão (respostas às questões)
5. Desafio Prático (data augmentation + limiar de segurança)
6. Conclusões
7. Referências

## 1. Introdução

Este relatório documenta o Laboratório 7, cujo tema é a transição do **processamento clássico de imagens**
(filtros espaciais projetados à mão, como Sobel e Gauss) para o **aprendizado profundo** com **Redes
Neurais Convolucionais (CNN)**, em que os filtros são **aprendidos a partir dos dados**. Construímos e
treinamos uma CNN compacta em **TensorFlow/Keras** para classificar as 10 classes do dataset **CIFAR-10**,
inspecionamos os *feature maps* aprendidos e discutimos a aplicação em **percepção robótica em tempo real**
(webcam), incluindo o compromisso entre **latência e acurácia** e o fenômeno de **domain shift**.

O objetivo de aprendizagem é compreender *o que* a rede aprende, *por que* a arquitetura convolucional é
adequada a imagens e *quais* as implicações práticas (segurança, hardware embarcado) de usar visão baseada
em CNN em um robô.

## 2. Fundamentação Teórica

### 2.1 Do filtro projetado ao filtro aprendido (Szeliski, cap. 5.4)
No processamento clássico, um **filtro espacial** é uma pequena matriz (*kernel*) de pesos **fixos**,
definidos manualmente para realçar um padrão específico. O filtro de **Sobel** para bordas verticais é

$$H=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}.$$

A operação de **convolução** desliza esse *kernel* sobre a imagem, calculando, em cada posição, a soma
ponderada da vizinhança. Uma **camada convolucional** de uma CNN faz exatamente a mesma operação, com uma
diferença crucial: **os pesos do kernel são parâmetros $\theta$ ajustáveis**, aprendidos por
**gradiente descendente** e **backpropagation** para minimizar uma função de perda. Em vez de *projetarmos*
o detector de bordas, a rede **descobre** os filtros mais úteis para a tarefa.

### 2.2 Blocos de uma CNN
- **`Conv2D`** — aplica $F$ *kernels* $K\times K\times C$ em paralelo, produzindo $F$ *feature maps*. Usa
  **compartilhamento de pesos** (o mesmo *kernel* percorre toda a imagem) → poucos parâmetros e
  **equivariância à translação**.
- **Ativação `ReLU`** — introduz não linearidade ($\max(0,x)$), permitindo aprender funções complexas.
- **`MaxPooling2D`** — reduz a resolução espacial (subamostragem) mantendo as ativações mais fortes; dá
  **invariância a pequenas translações** e reduz custo.
- **`Flatten` + `Dense`** — as camadas densas (totalmente conectadas) integram as características globais e
  tomam a **decisão** de classificação; a `softmax` final produz probabilidades por classe.
- **`Dropout`** — regularização: desliga aleatoriamente neurônios no treino, reduzindo *overfitting*.

### 2.3 Treinamento
A rede é treinada minimizando a **perda de entropia cruzada** entre a distribuição prevista (`softmax`) e o
rótulo verdadeiro, com o otimizador **Adam**. A **normalização** das entradas (pixels em $[0,1]$)
condiciona os gradientes e acelera/estabiliza a convergência. O acompanhamento das curvas de **perda** e
**acurácia** de treino vs. validação permite diagnosticar **overfitting** (treino melhora, validação
estaciona/piora).

### 2.4 Hierarquia de características e interpretabilidade
As camadas iniciais aprendem detectores de **bordas, cores e texturas**; as mais profundas, por
**composição** e **campo receptivo crescente**, representam **partes e formas** cada vez mais abstratas.
Inspecionar os *feature maps* (Szeliski 5.4.6) "abre a caixa-preta" e mostra que a rede reconstruiu, de
forma automática, detectores semelhantes aos filtros clássicos — e muitos outros.

### 2.5 Da bancada ao mundo físico
Ao levar o modelo para a **webcam**, surge o **domain shift**: as imagens reais diferem das de treino
(iluminação, escala, fundo), derrubando a acurácia. Além disso, cada inferência custa **tempo**; em um
robô móvel, a **latência** define quanto o robô anda "às cegas" entre capturar e decidir — um compromisso
direto entre **complexidade da CNN** e **segurança**.

## 3. Procedimentos Experimentais

Abaixo está o pipeline completo executado no Colab (GPU T4). Cada bloco corresponde a uma etapa do roteiro.
As respostas às questões estão na Seção 4.

In [ ]:
# 1. Importação de bibliotecas
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time

print(f"TensorFlow: {tf.__version__}")
print("GPU ativa:", len(tf.config.list_physical_devices('GPU')) > 0)

### 3.1 Convolução clássica (Sobel) — visualização da extração de características
Aplicamos o filtro de Sobel manualmente e, conforme pedido no roteiro, geramos **três exemplos por
integrante** (índices diferentes do CIFAR-10).

In [ ]:
# 2. Convolução clássica com OpenCV/NumPy
(x_train_raw, y_train_raw), _ = tf.keras.datasets.cifar10.load_data()

sobel_x = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)

def mostra_sobel(indices, titulo):
    plt.figure(figsize=(12, 4*len(indices)//3 if len(indices)>=3 else 4))
    for j, idx in enumerate(indices):
        img = x_train_raw[idx]
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        edges = cv2.filter2D(gray, -1, sobel_x)
        plt.subplot(len(indices), 2, 2*j+1); plt.imshow(img); plt.axis('off')
        plt.title(f"Original [{idx}]")
        plt.subplot(len(indices), 2, 2*j+2); plt.imshow(edges, cmap='gray'); plt.axis('off')
        plt.title("Sobel X (bordas verticais)")
    plt.suptitle(titulo); plt.tight_layout(); plt.show()

# Três imagens diferentes por aluno (ajuste os índices se quiser outras imagens)
mostra_sobel([0, 1, 2],   "Filtragem Sobel — Lucas")
mostra_sobel([10, 11, 12], "Filtragem Sobel — Pedro")
mostra_sobel([20, 21, 22], "Filtragem Sobel — Roberto")

### 3.2 Preparação do dataset CIFAR-10

In [ ]:
# 3. Download e pré-processamento
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0   # normalização para [0,1]
x_test  = x_test.astype('float32') / 255.0

class_names = ['Aviao','Automovel','Passaro','Gato','Cervo',
               'Cachorro','Sapo','Cavalo','Navio','Caminhao']

plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2,5,i+1); plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]]); plt.axis('off')
plt.tight_layout(); plt.show()

### 3.3 Arquitetura da CNN (RobotVisionNet)

In [ ]:
# 4. Definição do modelo Keras
def build_robot_cnn(input_shape=(32,32,3), num_classes=10):
    return models.Sequential([
        layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=input_shape,name='conv_1'),
        layers.MaxPooling2D((2,2),name='pool_1'),
        layers.Conv2D(64,(3,3),activation='relu',padding='same',name='conv_2'),
        layers.MaxPooling2D((2,2),name='pool_2'),
        layers.Flatten(name='flatten'),
        layers.Dense(128,activation='relu',name='fc_1'),
        layers.Dropout(0.3,name='dropout'),
        layers.Dense(num_classes,activation='softmax',name='output'),
    ])

model = build_robot_cnn()
model.summary()

### 3.4 Compilação e treinamento

In [ ]:
# 5. Treinamento (10 épocas)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=10,
                    validation_data=(x_test, y_test), batch_size=64)

### 3.5 Diagnóstico: curvas de aprendizado e matriz de confusão

In [ ]:
# 6. Curvas e matriz de confusão
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],label='Treino')
plt.plot(history.history['val_accuracy'],label='Validação')
plt.xlabel('Época'); plt.ylabel('Acurácia'); plt.title('Acurácia'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'],label='Treino')
plt.plot(history.history['val_loss'],label='Validação')
plt.xlabel('Época'); plt.ylabel('Perda'); plt.title('Perda'); plt.legend()
plt.show()

y_pred = np.argmax(model.predict(x_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predito'); plt.ylabel('Real'); plt.title('Matriz de Confusão'); plt.show()

print("Acurácia final (teste): %.3f" % (np.mean(y_pred == y_test.ravel())))

### 3.6 Visualização dos *feature maps* da camada `conv_1`

In [ ]:
# 7. Inspeção de ativações
model.build(input_shape=(None,32,32,3))
_ = model(tf.zeros((1,32,32,3)))
activation_model = tf.keras.Model(inputs=model.layers[0].input,
                                  outputs=model.get_layer('conv_1').output)
feature_maps = activation_model.predict(np.expand_dims(x_test[0], axis=0))

plt.figure(figsize=(12,6))
for i in range(16):
    plt.subplot(4,4,i+1); plt.imshow(feature_maps[0,:,:,i], cmap='viridis'); plt.axis('off')
    plt.title(f'Canal {i+1}')
plt.suptitle('Mapas de características aprendidos (conv_1)'); plt.show()

### 3.7 Medição do tempo de inferência (Parte 8, Questão 1)

In [ ]:
# Tempo de inferência de UM quadro
amostra = np.expand_dims(x_test[0], axis=0)
_ = model.predict(amostra)                 # 'aquecimento'
t0 = time.time()
for _ in range(50):
    _ = model.predict(amostra, verbose=0)
dt_ms = (time.time()-t0)/50*1000
print(f"Tempo médio de inferência: {dt_ms:.1f} ms/quadro  ->  {1000/dt_ms:.1f} FPS")
print("Orçamento a 30 FPS = 33 ms/quadro.  Cabe?", "SIM" if dt_ms < 33 else "NÃO (otimizar)")

### 3.8 Teste ao vivo com a webcam (Colab)
A célula de webcam é específica do Colab (ponte JavaScript↔Python). Rode-a no Colab, mostre objetos reais
(miniaturas de carro/avião/animal ou imagens no celular) e **cole aqui os prints** com a classe prevista e
a confiança.

**_(inserir prints da webcam: acertos e erros ao vivo)_**

## 4. Análise e Discussão — respostas às questões

**Q1 — Limitação de filtros codificados à mão (Sobel/Canny) num robô externo.**
Um *kernel* fixo responde a **um único padrão** (aqui, bordas verticais numa dada escala) e é **linear**:
não se adapta a variações de **iluminação, escala, rotação, ponto de vista** nem a **fundos com textura**.
Ele extrai apenas informação de **baixo nível** (bordas), sem qualquer **semântica** — não "sabe" o que é
um obstáculo. Em ambiente externo, sombras, brilho e clutter geram bordas espúrias e o filtro exige
**re-sintonia manual** para cada condição. Matematicamente, é uma convolução de pesos constantes: sem
adaptação aos dados e sem hierarquia de representação.

**Q2 — O que a camada convolucional "aprende".**
Aprende os **próprios pesos dos kernels** (parâmetros $\theta$) e os *bias*. Por gradiente descendente/
backpropagation, ajusta esses pesos para **minimizar a perda** da tarefa; assim a rede **descobre** quais
filtros são úteis (bordas em várias orientações, cores, texturas) em vez de nós os projetarmos. Ela aprende
uma **representação de características dirigida pelos dados**.

**Q3 — Por que normalizar dividindo por 255.**
Coloca os pixels em $[0,1]$, deixando as entradas em **escala pequena e homogênea**. Isso **condiciona os
gradientes** (evita ativações/gradientes muito grandes, que causam instabilidade e *exploding gradients*),
equilibra os passos do otimizador (Adam) e **acelera e estabiliza** a convergência.

**Q4 — CIFAR (32×32) vs. câmera 4K (3840×2160) e a primeira `Dense`.**
O número de entradas da primeira densa cresce com a **área** da imagem. No nosso modelo, após dois
*poolings* (32→16→8) com 64 canais, o `Flatten` tem $8\times8\times64=4096$ valores e a `Dense(128)` tem
$4096\times128+128\approx 5{,}2\times10^{5}$ pesos. Se uma imagem 4K fosse achatada e ligada diretamente a
uma `Dense(128)`: $3840\times2160\times3=24{,}88\times10^{6}$ entradas $\times128\approx 3{,}2\times10^{9}$
pesos **em uma única camada** — inviável em **hardware embarcado** (memória e latência explodem, além de
*overfitting*). Soluções: **redimensionar/subamostrar** a entrada, usar **global average pooling** ou redes
**totalmente convolucionais**.

**Q5 — Por que `Conv2D` no início e `Dense` no fim.**
As convolucionais **preservam a estrutura espacial** e extraem características locais com
**compartilhamento de pesos** (poucos parâmetros, equivariância à translação); o *pooling* adiciona
**invariância**. Só depois de obter boas características as **densas** integram a informação **global** e
**decidem** a classe. Inverter (densa primeiro) achataria os pixels crus, **perderia a localidade** e
explodiria o número de parâmetros. Papel: conv/pool = **extrator de características**; dense = **classificador**.

**Q6 — Função de perda e o cenário treino↓/validação↑.**
A **perda** mede a discrepância entre a distribuição prevista (`softmax`) e o rótulo verdadeiro (entropia
cruzada); é o que o treino minimiza. Se a perda de **treino cai rumo a zero** mas a de **validação
estaciona/sobe**, ocorre **overfitting**: a rede **memoriza** o treino e **não generaliza**. Na prática, o
robô acertaria as cenas de treino e **falharia em cenas novas** — perigoso. Mitiga-se com *dropout*,
*data augmentation*, regularização, *early stopping* e mais dados.

**Q7 — Duas classes com mais falsos positivos.**
_(Leia a SUA matriz de confusão e cite as duas classes.)_ Tipicamente, **Gato↔Cachorro** e
**Automóvel↔Caminhão** concentram os erros. Motivo semântico: são classes **visualmente semelhantes**
(forma, textura, contexto) e, em 32×32, os detalhes que as distinguem quase desaparecem.

**Q8 — Todos os erros têm o mesmo peso?**
Não. Confundir **Automóvel↔Caminhão** (ambos veículos) tem consequência **parecida** para a frenagem;
confundir **Cachorro→Caminhão** troca um **ser vivo** por um objeto e é **crítico** para a segurança. A
matriz de confusão permite **auditar erro a erro** e aplicar uma avaliação **sensível ao custo**
(ponderar cada erro pelo risco real), em vez de olhar só a acurácia global.

**Q9 — *Feature maps* profundos: simples ou complexos?**
Mais **complexos/abstratos** (partes e formas do objeto), não linhas e pontos. Motivo "anatômico" da CNN:
**hierarquia** (cada camada combina as características da anterior) somada ao **campo receptivo crescente**
e às **não linearidades** — assim, camadas profundas "enxergam" regiões maiores e representações mais
semânticas.

**Q10 — Três fatores de *domain shift* da webcam vs. CIFAR-10.**
(1) **Iluminação** (temperatura de cor, sombras, brilho) diferente da do dataset; (2) **escala/resolução**
— a webcam é de alta resolução e o objeto aparece em escala/posição variáveis, enquanto o CIFAR é 32×32 com
o objeto centralizado; (3) **fundo complexo** (cena real bagunçada) contra o fundo limpo das amostras.
Outros: desfoque de movimento, balanço de branco/cor da câmera, ângulo de visão.

**Q11 — Distância "às cegas" a 2 m/s com 500 ms de pipeline.**
$d = v\cdot t = 2\,\text{m/s}\times0{,}5\,\text{s}= \mathbf{1{,}0\ m}$. Ou seja, o robô percorre **1 metro**
entre capturar a imagem e decidir. Quanto **mais complexa** a CNN, **maior a latência** e **maior a
distância cega** — um compromisso direto entre **acurácia** e **segurança mecânica**; a latência precisa
ser limitada por requisito de projeto.

---
### Parte 8 — Questões adicionais

**Q1 (latência × acurácia).** A 30 FPS há **33 ms** por ciclo. O tempo de inferência **medido** foi
_(inserir o valor impresso pela célula 3.7, em ms)_ → **cabe / não cabe** no orçamento. Se não couber:
modelo mais leve, resolução menor ou GPU.

**Q2 (a "ilusão" da acurácia).** Ao vivo a acurácia cai por **iluminação**, **fundo complexo** e
**mudança de escala**. **_(inserir uma imagem/print ilustrando cada um dos três fatores)_**.

## 5. Desafio Prático (Parte 9)

### 5.1 Data augmentation (robustez a orientação/iluminação)
Reconstruímos o modelo com uma etapa de aumento de dados na entrada e retreinamos, comparando com o
baseline.

In [ ]:
# Data augmentation + retreino
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

def build_cnn_aug(input_shape=(32,32,3), num_classes=10):
    return models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(32,(3,3),activation='relu',padding='same',name='conv_1'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64,(3,3),activation='relu',padding='same',name='conv_2'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128,activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes,activation='softmax'),
    ])

model_aug = build_cnn_aug()
model_aug.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_aug = model_aug.fit(x_train, y_train, epochs=10,
                         validation_data=(x_test, y_test), batch_size=64)
print("Acurácia val (baseline) : %.3f" % max(history.history['val_accuracy']))
print("Acurácia val (aug)      : %.3f" % max(hist_aug.history['val_accuracy']))

### 5.2 Filtro de rejeição por confiança (*safety threshold*)
Se a maior probabilidade for **< 0,60**, o sistema declara **"Objeto Não Identificado / Incerto"** em vez
de arriscar uma ação. Abaixo, a função de decisão segura usada também na classificação da webcam.

In [ ]:
LIMIAR_SEGURANCA = 0.60

def classificar_seguro(model, img_norm):
    \"\"\"img_norm: array 32x32x3 em [0,1]. Retorna (rotulo, confianca).\"\"\"
    p = model.predict(np.expand_dims(img_norm, axis=0), verbose=0)[0]
    i = int(np.argmax(p)); conf = float(p[i])
    if conf < LIMIAR_SEGURANCA:
        return "Objeto Nao Identificado / Incerto", conf
    return class_names[i], conf

# Exemplo com uma imagem de teste:
rotulo, conf = classificar_seguro(model, x_test[0])
print(f"Decisão: {rotulo}  (confiança {conf:.2f})")

## 6. Conclusões

O laboratório evidenciou, na prática, a diferença entre **filtros projetados** e **filtros aprendidos**: a
CNN treinada em CIFAR-10 reconstruiu automaticamente detectores de borda/textura (visíveis nos *feature
maps*) e atingiu acurácia _(inserir valor)_ no conjunto de teste, superando qualquer combinação manual de
filtros. A **matriz de confusão** revelou que os erros se concentram em classes visualmente próximas, e a
discussão de **custo dos erros** mostrou por que, em robótica, a avaliação deve ser **sensível ao risco**,
não apenas à acurácia. O teste ao vivo expôs o **domain shift** (iluminação, fundo e escala) e a medição de
**latência** conectou complexidade do modelo à **segurança** (a ~2 m/s, 500 ms = 1 m às cegas). O
*data augmentation* e o **limiar de confiança** foram implementados como melhorias diretas de robustez e
segurança. _(Comente os números reais que vocês obtiveram.)_

## 7. Referências
1. SZELISKI, R. *Computer Vision: Algorithms and Applications*. 2ª ed., cap. 5.4.
2. TensorFlow/Keras — documentação oficial. https://www.tensorflow.org/
3. CIFAR-10 dataset — Krizhevsky, A. *Learning Multiple Layers of Features from Tiny Images*, 2009.
4. Roteiro do Laboratório 7 — ESZA019 (2026.2), Prof. Celso S. Kurashima.
5. CNPq. Portaria 2664/2026 (integridade e uso de IAG).